# L3c Example: Recursive Implementation of Fibonacci Sequence Calculation
In this example, we illustrate recursion concepts by benchmarking three implementations of the Fibonacci sequence computation using [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
> * __Measure instead of assuming:__ Benchmark competing implementations of one calculation at the same problem size, rather than reasoning from intuition about which formulation ought to be faster.
> * __Find the repeated work in a call tree:__ Explain why a naive recursive definition recomputes the same subproblems, and describe how that repetition grows as the input grows.
> * __Apply memoization deliberately:__ Cache each completed subproblem so it is computed once, and say what that trades away in memory to buy back in time.

* __Case 1: Vanilla loop-based implementation:__ First, we benchmark the time required to calculate the sequence $F_{0},\dots,F_{n}$ using the for-loop implementation, `fibonacci(n::Int64)::Dict{Int64, Int64}`.
* __Case 2: Standard recursive implementation:__ Next, a plain recursive implementation, `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64`. It writes every index it visits into the `series` dictionary it is handed, and also returns $F_{n}$.
* __Case 3: Memoized recursive implementation:__ Lastly, the same recursion with memoization, `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64`, which checks `series` before recomputing a subproblem.

We expect the memoized version to beat the plain recursion by a wide margin, because memoization removes redundant calculation rather than merely speeding it up. Let's see whether the measurements agree.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 


Let's set up the computational environment.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # what is this doing?

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This example does not need it. Everything below uses `Base`, [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), and [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

### Implementations
The three versions we benchmark live in [the `Compute.jl` file](src/Compute.jl), which [`Include.jl`](Include.jl) loads for us.

* __Vanilla loop-based implementation:__ `fibonacci(n::Int64)::Dict{Int64, Int64}` fills a dictionary from `0` to `n` with a single for-loop.
* __Standard recursive implementation:__ `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` computes $F_{n}$ straight from the recurrence. It stores each index it visits in `series`, but it does not consult `series` before recursing, so the same subproblems are recomputed many times.
* __Memoized recursive implementation:__ `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` is the same recursion with one extra line: it returns a stored value when it finds one, so each subproblem is computed exactly once.

All three mutate or return through `series` and also return $F_{n}$ as an `Int64`.


### Constants
Let's set the constants used below. `correct_fibonacci_sequence` holds the values we check each implementation against, and `benchmark_index` fixes the problem size, which every case shares so that the three timings can be compared to each other.

In [ ]:
correct_fibonacci_sequence = Dict(0 => 0, 1 => 1, 2 => 1, 3 => 2, 4 => 3, 5 => 5, 6 => 8, 7 => 13,
                                  8 => 21, 9 => 34, 10 => 55, 11 => 89, 12 => 144, 13 => 233,
                                  14 => 377, 15 => 610); # F0 through F15, so 16 known values
benchmark_index = 25; # every case below is benchmarked at this same n, so the timings are comparable

___

## Case 1: Test the for loop implementation of Fibonacci computation
Let's use the [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) to compute the average time required to calculate the sequence $F_{0},\dots,F_{n}$ using the vanilla implementation of the `fibonacci` function (for-loop-based implementation). However, before we benchmark the for loop implementation, let's check that it is correct by [using the `@test` macro exported by the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/).

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = fibonacci(number_of_test_terms);
    
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

Now that we have verified correctness, let's benchmark the for-loop implementation of the Fibonacci sequence calculation.

> __Benchmarking:__ The [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) exports the [@benchmarkable macro](https://juliaci.github.io/BenchmarkTools.jl/stable/reference/#BenchmarkTools.@benchmarkable-Tuple), which computes a function's runtime and memory profile. It runs the function many times and returns statistical information about its performance. We fix `samples` and `evals` explicitly rather than calling `tune!`, so all three cases are measured the same way; the reason `evals = 1` matters shows up in Case 3.

How does the vanilla implementation perform? Let's find out!

In [ ]:
result_basal = let
    test_run_basal = @benchmarkable fibonacci($(benchmark_index));
    result_basal = run(test_run_basal; samples = 200, evals = 1)
end

## Case 2: Test the recursive implementation of the Fibonacci computation
Next, let's benchmark a recursive implementation. The `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Nothing` function is a mutating recursive function that computes the sequence $F_{0},\dots, F_{n}$ for a given $n$. The recursive sequence is stored in the `series::Dict{Int64, Int64}` argument. This takes advantage of [the mutating function behavior](https://docs.julialang.org/en/v1/manual/functions/#man-argument-passing) in Julia, which allows us to update the dictionary in place without returning a new dictionary.

Let's verify that the recursive implementation is correct by checking that it computes the Fibonacci sequence correctly for $F_{0},\dots,F_{n}$, where $n$ is the number of terms in the `correct_fibonacci_sequence` dictionary.

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = Dict{Int64, Int64}(); # initialize an empty dictionary
    fibonacci!(number_of_test_terms, my_computed_sequence); # mutates the dictionary in place; the returned Fn is ignored here
    
    # verify correctness - for terms 0 ... number_of_test_terms
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

No testing explosions? Great! Then how does the recursive implementation perform relative to the baseline implementation of the Fibonacci computation?

In [ ]:
result_recursive = let
    test_run_recursive = @benchmarkable fibonacci!($(benchmark_index), series) setup=(series = Dict{Int,Int}())
    result_recursive = run(test_run_recursive; samples = 200, evals = 1)
end

__Hmmmm.__ Wow! At the same `benchmark_index`, the plain recursion is __far slower__ than the loop. The cause is not the cost of a function call. It is that `fibonacci!(...)` never consults `series` before recursing, so it recomputes $F_{n-1}$ and $F_{n-2}$ from scratch at every level, and the number of calls grows exponentially in `n`. Lesson learned: recursion is not automatically slower than iteration, but a recursion that recomputes its own subproblems certainly is.

The picture below is the reason. Each node is a call, and the plain recursion regrows the whole shaded subtree every time it needs a value it has already computed once. Memoization is the observation that the second visit to a node can be answered from a table.

<div>
    <center>
      <img
        src="figs/Fig-Fibonacci-Recursive.svg"
        alt="Call tree for the recursive Fibonacci computation, showing repeated subtrees"
        height="400"
        width="800"
      />
    </center>
  </div>

## Case 3: Test the recursive implementation of the Fibonacci computation with memoization
Finally, let's benchmark a recursive Fibonacci function that uses memoization. The `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Nothing` implementation is a mutating recursive function that uses memoization to speed up the computation of the sequence $F_{0},\dots, F_{n}$ for a given $n$. The recursive sequence is stored in the `series::Dict{Int64, Int64}` argument.

First, does this implementation do what we expect? Let's verify that it computes the Fibonacci sequence correctly for $F_{0},\dots,F_{n}$, where $n$ is the number of terms in the `correct_fibonacci_sequence` dictionary.

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = Dict{Int64, Int64}(); # initialize an empty dictionary
    memoization_fibonacci!(number_of_test_terms, my_computed_sequence); # mutates the dictionary in place; the returned Fn is ignored here
    
    # verify correctness - for terms 0 ... number_of_test_terms
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

Does memoization change the runtime and allocation profile of the recursive implementation?

> __Two guards make this measurement honest, and both are necessary.__ The `setup=` argument gives each sample a __fresh__ empty dictionary; without it a single shared dictionary would be populated once and every later sample would find the answer already cached. But `setup=` runs once per _sample_, not once per _evaluation_, so `evals = 1` is also required: left to tune itself, BenchmarkTools would run many evaluations against one setup and we would be back to timing a warm cache. Together they measure what we actually want, which is the cost of memoizing from cold. The same pair of guards is on Case 2.

In [ ]:
result_recursive_memo = let
    test_run_recursive_memo = @benchmarkable memoization_fibonacci!($(benchmark_index), series) setup=(series = Dict{Int,Int}())
    result_recursive_memo = run(test_run_recursive_memo; samples = 200, evals = 1)
end

## Summary
Three implementations of one calculation, benchmarked at the same problem size, separate the cost of recursion from the cost of repeated work.

> __Key Takeaways:__
>
> * **Recursion is not inherently slow:** The naive recursive Fibonacci loses to a loop because its call tree recomputes the same subproblems, not because a function call is expensive, which is why the fix is to remove the repetition rather than to abandon recursion.
> * **Memoization changes the growth rate, not the constant:** Storing each completed subproblem turns an exponential number of calls into a linear one, so it wins by orders of magnitude rather than by a few percent.
> * **A benchmark is only as honest as its setup:** Timing two implementations at different problem sizes, or reusing a warm cache across samples, produces real numbers that answer a different question than the one being asked.

Choose the algorithm from the structure of the problem, and remember that an elegant recursive statement sometimes needs memoization before it is practical.
___